# Import Libraries #

In [12]:
import pandas as pd
import numpy as np
import logging

# Set Up Logging #

In [23]:
logging.basicConfig(level=logging.INFO)

# Read CSV Files #

In [ ]:
def read_data(call_logs_path, agent_roster_path, disposition_summary_path):
    
    call_logs = pd.read_csv("Data/call_logs.csv")
    agent_roster = pd.read_csv("Data/agent_roster.csv")
    disposition_summary = pd.read_csv("Data/disposition_summary.csv")

    return call_logs, agent_roster, disposition_summary

# Validate Data #

In [24]:
def validate_data(call_logs, agent_roster, disposition_summary):
    
    # Check for missing values
    for df in [call_logs, agent_roster, disposition_summary]:
        if df.isnull().values.any():
            logging.warning("Missing values found in dataframe.")
    
    # Check for duplicates
    for df in [call_logs, agent_roster, disposition_summary]:
        if df.duplicated().any():
            logging.warning("Duplicate entries found in dataframe.")
    
    # Validate date formats
    call_logs['call_date'] = pd.to_datetime(call_logs['call_date'], errors='coerce')
    disposition_summary['call_date'] = pd.to_datetime(disposition_summary['call_date'], errors='coerce')
    
    return call_logs, agent_roster, disposition_summary

# Join Logic #

In [25]:
def merge_data(call_logs, agent_roster, disposition_summary):

    # Merge call logs with disposition summary
    merged_data = call_logs.merge(disposition_summary, on=['agent_id', 'org_id', 'call_date'], how='outer')
    
    # Merge with agent roster
    final_data = merged_data.merge(agent_roster, on=['agent_id', 'org_id'], how='outer')
    
    # Handle mismatches
    if final_data.isnull().values.any():
        logging.info("Some entries did not match during the merge.")
    
    return final_data

# Feature Engineering #

In [ ]:
def compute_metrics(final_data):
    
    # Group by agent_id and call_date
    summary = final_data.groupby(['agent_id', 'call_date']).agg(
        total_calls=('call_id', 'count'),
        unique_loans_contacted=('installment_id', 'nunique'),
        completed_calls=('status', lambda x: (x == 'completed').sum()),
        avg_call_duration=('duration', lambda x: x.mean() / 60),  # Convert to minutes
        presence=('login_time', lambda x: 1 if x.notnull().any() else 0)
    ).reset_index()
    
    # Calculate connect rate
    summary['connect_rate'] = summary['completed_calls'] / summary['total_calls']
    
    return summary

# Output #

In [18]:
def save_report(summary, output_path):
    summary.to_csv(output_path, index=False)
    logging.info(f"Report saved to {output_path}")

# Generate Slack-style Summary Message

In [ ]:
def generate_summary_message(summary):
    
    top_performer = summary.loc[summary['connect_rate'].idxmax()]
    total_active_agents = summary['agent_id'].nunique()
    average_duration = summary['avg_call_duration'].mean()
    
    message = (
        f"Agent Summary for {summary['call_date'].iloc[0].date()}\n"
        f"Top Performer: {top_performer['agent_id']} ("
        f"{top_performer['connect_rate'] * 100:.2f}% connect rate)\n"
        f"Total Active Agents: {total_active_agents}\n"
        f"Average Duration: {average_duration:.2f} min"
    )
    
    return message


#  Main Function #

In [ ]:
def main(call_logs_path, agent_roster_path, disposition_summary_path, output_path):

    call_logs, agent_roster, disposition_summary = read_data(call_logs_path, agent_roster_path, disposition_summary_path)
    call_logs, agent_roster, disposition_summary = validate_data(call_logs, agent_roster, disposition_summary)
    final_data = merge_data(call_logs, agent_roster, disposition_summary)
    summary = compute_metrics(final_data)

    save_report(summary, output_path)
    
    message = generate_summary_message(summary)
    print(message)

# Example usage
if __name__ == "__main__":
    main('MultipleFiles/call_logs.csv', 'MultipleFiles/agent_roster.csv', 'MultipleFiles/disposition_summary.csv', 'agent_performance_summary.csv')


INFO:root:Some entries did not match during the merge.
INFO:root:Report saved to agent_performance_summary.csv


Agent Summary for 2025-04-28
Top Performer: A003 (38.10% connect rate)
Total Active Agents: 20
Average Duration: 0.13 min
